# 70 — LGBM: CRC + Single-Conc + ChEMBL PXR (3-way)

Combines all three PXR-specific data sources: CRC dose-response (weight=1.0), single-conc FDR-weighted pseudo-pEC50 (weight varies), ChEMBL PXR (weight=0.75).


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED,
    verbose=-1, n_jobs=4,
)


In [2]:
def full_metrics(y_true, y_pred, cliff_pairs_df=None, label=""):
    """RAE, MAE, R², Pearson, Spearman, Kendall, Cliff_accuracy."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    mae_v  = float(np.mean(np.abs(yt - yp)))
    rae_v  = mae_v / float(np.mean(np.abs(yt - yt.mean()))) if yt.std() > 0 else float("nan")
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2_v   = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    pr_v, _ = stats.pearsonr(yt, yp)
    sp_v, _ = stats.spearmanr(yt, yp)
    kt_v, _ = stats.kendalltau(yt, yp)

    m = dict(RAE=rae_v, MAE=mae_v, R2=r2_v,
             Pearson=pr_v, Spearman=sp_v, Kendall=kt_v)

    if cliff_pairs_df is not None and len(cliff_pairs_df) > 0:
        correct = total = 0
        for _, row in cliff_pairs_df.iterrows():
            ia, ii = int(row.get("idx_active", -1)), int(row.get("idx_inactive", -1))
            if 0 <= ia < len(yp) and 0 <= ii < len(yp):
                correct += int(yp[ia] > yp[ii])
                total   += 1
        m["Cliff_acc"] = correct / total if total else float("nan")

    if label:
        cliff_str = f"  Cliff_acc={m.get('Cliff_acc', float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f}  MAE={mae_v:.4f}  R²={r2_v:.4f}  "
              f"Pearson={pr_v:.4f}  Spearman={sp_v:.4f}  Kendall={kt_v:.4f}{cliff_str}")
    return m


In [3]:
tr = load_train()
te = load_test()
print(f"CRC train: {len(tr):,}  |  Test: {len(te):,}")

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
active_mask = y_tr >= 5.5
print(f"X_tr: {X_tr.shape}  actives: {active_mask.sum()}")

cliff_pairs = (pd.read_parquet(DATA_PROCESSED / "cliff_pairs.parquet")
               if (DATA_PROCESSED / "cliff_pairs.parquet").exists()
               else pd.DataFrame())
print(f"Cliff pairs available: {len(cliff_pairs)}")


CRC train: 4,139  |  Test: 513


X_tr: (4139, 2265)  actives: 380
Cliff pairs available: 149


In [4]:
def run_cv(X_int, y_int, splits, X_ext=None, y_ext=None, w_ext=None,
           label="", params=LGBM_PARAMS):
    oof = np.full(len(y_int), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        Xf = X_int[tr_idx]; yf = y_int[tr_idx]
        Xv = X_int[va_idx]; yv = y_int[va_idx]
        wf = np.ones(len(yf), dtype=np.float32)
        if X_ext is not None and len(X_ext) > 0:
            Xf = np.vstack([Xf, X_ext])
            yf = np.concatenate([yf, y_ext])
            wf = np.concatenate([wf, w_ext if w_ext is not None
                                  else np.ones(len(y_ext), dtype=np.float32)])
        m = lgb.train(params, lgb.Dataset(Xf, label=yf, weight=wf),
                      valid_sets=[lgb.Dataset(Xv, label=yv)],
                      callbacks=[lgb.early_stopping(50, verbose=False),
                                 lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(Xv)
        print(f"  fold {fold+1}  val_RAE={rae(yv, oof[va_idx]):.4f}", flush=True)
    m_all    = full_metrics(y_int, oof, cliff_pairs, label=label)
    m_active = full_metrics(y_int[active_mask], oof[active_mask],
                            label=f"{label} [active≥5.5]")
    return oof, m_all, m_active


def train_final_and_predict(X_tr_all, y_tr_all, w_tr_all, X_te, params=LGBM_PARAMS):
    m = lgb.train(params, lgb.Dataset(X_tr_all, label=y_tr_all, weight=w_tr_all),
                  callbacks=[lgb.log_evaluation(-1)])
    return np.clip(m.predict(X_te), y_tr_all.min() - 0.5, y_tr_all.max() + 0.5)


In [5]:
from pxr.data import load_single_conc
from pxr.chem import to_inchikey, standardize_smiles
tr_iks = set(tr["smiles"].map(to_inchikey))

# 1. Single-conc
sp = load_single_conc()
sp["ik"] = sp["smiles"].map(to_inchikey)
sp_novel = sp[~sp["ik"].isin(tr_iks)].copy()
slope, intercept = 0.496, 5.10
sp_novel["pec50_pseudo"] = (intercept + slope * sp_novel["log2_fc_estimate"].clip(-6,6)).clip(3.0,7.5)
sp_w = np.clip(1.0 - sp_novel["fdr_bh"].fillna(1.0), 0.05, 1.0).values * 0.6 if "fdr_bh" in sp_novel else 0.3

# 2. ChEMBL PXR
chembl = pd.read_parquet(DATA_EXTERNAL / "chembl_nr_extended.parquet")
chembl = chembl[chembl["target_name"] == "PXR"].copy()
chembl["std_smi"] = chembl["smiles"].map(standardize_smiles)
chembl = chembl.dropna(subset=["std_smi"])
chembl["ik"] = chembl["std_smi"].map(to_inchikey)
chembl_novel = chembl[~chembl["ik"].isin(tr_iks)].drop_duplicates("ik")

# Combine
ext_smiles = list(sp_novel["smiles"]) + list(chembl_novel["std_smi"])
ext_y = np.concatenate([
    sp_novel["pec50_pseudo"].values.astype(np.float64),
    chembl_novel["pec50"].clip(3.5,9.0).values.astype(np.float64)
])
ext_w = np.concatenate([
    sp_w.astype(np.float32) if hasattr(sp_w,"__len__") else np.full(len(sp_novel), sp_w, dtype=np.float32),
    np.full(len(chembl_novel), 0.75, dtype=np.float32)
])
X_ext = impute(combined(ext_smiles))
y_ext, w_ext = ext_y, ext_w
W_EXT = float(ext_w.mean())
print(f"SP+ChEMBL_PXR external: {len(y_ext):,}  "
      f"(SP: {len(sp_novel):,}, ChEMBL: {len(chembl_novel):,})")


SP+ChEMBL_PXR external: 16,185  (SP: 15,284, ChEMBL: 901)


In [6]:
print("Running scaffold 5-fold CV...")
oof, m_all, m_active = run_cv(
    X_tr, y_tr, splits,
    X_ext=X_ext if len(X_ext) > 0 else None,
    y_ext=y_ext if len(X_ext) > 0 else None,
    w_ext=w_ext if len(X_ext) > 0 else None,
    label="CRC+SP+ChEMBL_PXR"
)
print(f"\nAugmented with {len(X_ext):,} external rows (weight scale={W_EXT:.2f})" if len(X_ext) > 0
      else "\nNo external augmentation (data empty)")

results_df = pd.DataFrame([m_all, m_active], index=["overall", "active≥5.5"])
print("\n" + results_df.round(4).to_string())


Running scaffold 5-fold CV...


  fold 1  val_RAE=0.8296


  fold 2  val_RAE=0.7789


  fold 3  val_RAE=0.7729


  fold 4  val_RAE=0.7428


  fold 5  val_RAE=0.7738


  [CRC+SP+ChEMBL_PXR] RAE=0.7771  MAE=0.7070  R²=0.1921  Pearson=0.6319  Spearman=0.6245  Kendall=0.4395  Cliff_acc=nan
  [CRC+SP+ChEMBL_PXR [active≥5.5]] RAE=2.7558  MAE=0.5779  R²=-4.5091  Pearson=0.1535  Spearman=0.0413  Kendall=0.0288

Augmented with 16,185 external rows (weight scale=0.45)

               RAE     MAE      R2  Pearson  Spearman  Kendall  Cliff_acc
overall     0.7771  0.7070  0.1921   0.6319    0.6245   0.4395        NaN
active≥5.5  2.7558  0.5779 -4.5091   0.1535    0.0413   0.0288        NaN


In [7]:
# Final model on all data
w_base = np.ones(len(y_tr), dtype=np.float32)
if len(X_ext) > 0:
    X_all = np.vstack([X_tr, X_ext])
    y_all = np.concatenate([y_tr, y_ext])
    w_all = np.concatenate([w_base, w_ext])
else:
    X_all, y_all, w_all = X_tr, y_tr, w_base

te_preds = train_final_and_predict(X_all, y_all, w_all, X_te)
np.save(DATA_PROCESSED / "oof_lgbm_crc_sp_chembl_pxr.npy", oof)
np.save(DATA_PROCESSED / "te_oof_lgbm_crc_sp_chembl_pxr.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub) == 513 and sub["pEC50"].notna().all()
out = SUBMISSIONS / "70_lgbm_crc_sp_chembl_pxr.csv"
sub.to_csv(out, index=False)
print(f"Saved {out}")
print(f"Test preds  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}")



Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\70_lgbm_crc_sp_chembl_pxr.csv
Test preds  min=3.49  median=5.26  max=6.05
